In [ ]:
import os
import sys
os.chdir("..")
sys.path.append(os.getcwd())
import pandas as pd
import duckdb
from pathlib import Path
from config import DATA_ROOT, NSE_DB_PATH

In [ ]:
RESEARCH_DB_PATH = DATA_ROOT / "research.db"
con = duckdb.connect(RESEARCH_DB_PATH)
con.execute(f"ATTACH '{NSE_DB_PATH}' AS nse (READ_ONLY)") 

In [3]:
# read from nse, write to research.db
con.execute("""
    CREATE OR REPLACE TABLE forward_returns AS
    WITH daily AS (
        SELECT
            trade_date,
            index_name,
            close
        FROM nse.market_activity_index
        WHERE index_name = 'Nifty 50'   -- adjust to exact name in your data
        ORDER BY trade_date
    )
    SELECT
        d.trade_date,
        d.index_name,
        d.close,

        -- forward returns
        ROUND((f1.close - d.close) / d.close * 100, 4) AS fwd_ret_1d,
        ROUND((f5.close - d.close) / d.close * 100, 4) AS fwd_ret_5d,
        ROUND((f20.close - d.close) / d.close * 100, 4) AS fwd_ret_20d,

        -- direction label (for classification)
        CASE WHEN f1.close > d.close THEN 1 ELSE 0 END AS up_1d

    FROM daily d
    LEFT JOIN daily f1
        ON f1.trade_date = (
            SELECT MIN(trade_date) FROM daily
            WHERE trade_date > d.trade_date
        )
    LEFT JOIN daily f5
        ON f5.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 4
        )
    LEFT JOIN daily f20
        ON f20.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 19
        )
    ;
""")

In [4]:
results = con.execute(""" SELECT * FROM forward_returns ORDER BY trade_date DESC LIMIT 10 OFFSET 20; """).fetchall()
for row in results:
    print(row)

(datetime.date(2026, 5, 22), 'Nifty 50', 23719.3, 1.3171, -1.4195, 1.6172, 1)
(datetime.date(2026, 5, 21), 'Nifty 50', 23654.7, 0.2731, -0.4521, 1.5151, 1)
(datetime.date(2026, 5, 20), 'Nifty 50', 23659.0, -0.0182, 1.0489, 2.1514, 0)
(datetime.date(2026, 5, 19), 'Nifty 50', 23618.0, 0.1736, 1.252, 1.9803, 1)
(datetime.date(2026, 5, 18), 'Nifty 50', 23649.95, -0.1351, 1.6142, 1.4343, 0)
(datetime.date(2026, 5, 15), 'Nifty 50', 23643.5, 0.0273, 0.3206, 0.8899, 1)
(datetime.date(2026, 5, 14), 'Nifty 50', 23689.6, -0.1946, -0.1473, -0.2816, 0)
(datetime.date(2026, 5, 13), 'Nifty 50', 23412.6, 1.1831, 1.0524, -1.0721, 1)
(datetime.date(2026, 5, 12), 'Nifty 50', 23379.55, 0.1414, 1.0199, -0.704, 1)
(datetime.date(2026, 5, 11), 'Nifty 50', 23815.85, -1.832, -0.6966, -2.4091, 0)


In [5]:
results = con.execute(""" SELECT DISTINCT index_name FROM nse.market_activity_index ORDER BY 1; """).fetchall()
for row in results:
    print(row)

('BHARATBOND-APR25',)
('BHARATBOND-APR30',)
('BHARATBOND-APR31',)
('BHARATBOND-APR32',)
('BHARATBOND-APR33',)
('India VIX',)
('NIFTY Alpha 50',)
('NIFTY AlphaLowVol',)
('NIFTY CONSR DURBL',)
('NIFTY HEALTHCARE',)
('NIFTY IND DIGITAL',)
('NIFTY INDIA MFG',)
('NIFTY LARGEMID250',)
('NIFTY M150 QLTY50',)
('NIFTY MICROCAP250',)
('NIFTY MID SELECT',)
('NIFTY MIDCAP 100',)
('NIFTY MIDCAP 150',)
('NIFTY MIDSML 400',)
('NIFTY OIL AND GAS',)
('NIFTY SMLCAP 100',)
('NIFTY SMLCAP 250',)
('NIFTY SMLCAP 50',)
('NIFTY TOTAL MKT',)
('NIFTY100 EQL Wgt',)
('NIFTY100 ESG',)
('NIFTY100 LowVol30',)
('NIFTY100 Qualty30',)
('NIFTY200 QUALTY30',)
('NIFTY50 EQL Wgt',)
('NIFTY500 MULTICAP',)
('Nifty 100',)
('Nifty 200',)
('Nifty 50',)
('Nifty 500',)
('Nifty AQL 30',)
('Nifty AQLV 30',)
('Nifty Auto',)
('Nifty Bank',)
('Nifty CPSE',)
('Nifty Capital Mkt',)
('Nifty Cement',)
('Nifty Chemicals',)
('Nifty Commodities',)
('Nifty Consumption',)
('Nifty CoreHousing',)
('Nifty Corp MAATR',)
('Nifty Div Opps 50',)
('Ni

In [6]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close         AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close        AS vix_close

FROM forward_returns fr
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

ORDER BY fr.trade_date
""")

In [7]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10 OFFSET 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 5, 22), 23719.3, 1.3171, -1.4195, 1.6172, 1, 17.91)
(datetime.date(2026, 5, 21), 23654.7, 0.2731, -0.4521, 1.5151, 1, 17.8225)
(datetime.date(2026, 5, 20), 23659.0, -0.0182, 1.0489, 2.1514, 0, 18.44)
(datetime.date(2026, 5, 19), 23618.0, 0.1736, 1.252, 1.9803, 1, 18.675)
(datetime.date(2026, 5, 18), 23649.95, -0.1351, 1.6142, 1.4343, 0, 19.63)
(datetime.date(2026, 5, 15), 23643.5, 0.0273, 0.3206, 0.8899, 1, 18.79)
(datetime.date(2026, 5, 14), 23689.6, -0.1946, -0.1473, -0.2816, 0, 18.6125)
(datetime.date(2026, 5, 13), 23412.6, 1.1831, 1.0524, -1.0721, 1, 19.425)
(datetime.date(2026, 5, 12), 23379.55, 0.1414, 1.0199, -0.704, 1, 19.28)
(datetime.date(2026, 5, 11), 23815.85, -1.832, -0.6966, -2.4091, 0, 18.5525)


In [8]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 13 THEN '1_low <13'
            WHEN vix_close < 16 THEN '2_calm 13-16'
            WHEN vix_close < 20 THEN '3_normal 16-20'
            WHEN vix_close < 25 THEN '4_elevated 20-25'
            ELSE                     '5_fear >25'
        END AS vix_regime,

        COUNT(*)                        AS days,
        ROUND(AVG(fwd_ret_1d), 3)       AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)       AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)      AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)      AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL

    GROUP BY 1
    ORDER BY 1
""").fetchall()

for row in results: print(row)

('1_low <13', 176, -0.008, -0.158, -0.466, 51.1)
('2_calm 13-16', 208, -0.018, 0.018, -0.22, 49.0)
('3_normal 16-20', 85, -0.075, -0.114, -0.069, 49.4)
('4_elevated 20-25', 20, 0.635, 1.275, 3.694, 65.0)
('5_fear >25', 6, 0.524, 4.01, 6.276, 83.3)


In [9]:
results = con.execute("""
    SELECT *
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
    ORDER BY trade_date DESC, expiry ASC
    LIMIT 20
""").fetchall()
for row in results: print(row)

('IDO', 'NIFTY', datetime.date(2026, 6, 23), datetime.date(2026, 6, 22), 144992250.0, 166337340.0, 0.8716758967048529, 24102.9, 24100.0)
('IDO', 'NIFTY', datetime.date(2026, 6, 30), datetime.date(2026, 6, 22), 96456070.0, 96037970.0, 1.004353486438749, 24102.90000000011, 24100.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 7), datetime.date(2026, 6, 22), 4735250.0, 4647630.0, 1.0188526195071466, 24102.89999999999, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 14), datetime.date(2026, 6, 22), 822510.0, 880360.0, 0.934288245717661, 24102.899999999987, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 21), datetime.date(2026, 6, 22), 110890.0, 76960.0, 1.4408783783783783, 24102.89999999994, 24100.0)
('IDO', 'NIFTY', datetime.date(2026, 7, 28), datetime.date(2026, 6, 22), 20946055.0, 17340895.0, 1.2078993039286612, 24102.90000000001, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 25), datetime.date(2026, 6, 22), 4691635.0, 3188315.0, 1.4715092454791951, 24102.9, 24000.0)
('IDO', 'NIFTY', d

In [10]:
results = con.execute("""
    SELECT
        expiry,
        trade_date,
        pe_oi,
        ce_oi,
        pe_oi + ce_oi AS total_oi
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND trade_date = '2026-06-10'
    ORDER BY total_oi DESC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 16), datetime.date(2026, 6, 10), 93749955.0, 114460970.0, 208210925.0)
(datetime.date(2026, 6, 30), datetime.date(2026, 6, 10), 69249000.0, 66301370.0, 135550370.0)
(datetime.date(2026, 12, 29), datetime.date(2026, 6, 10), 11812480.0, 9371595.0, 21184075.0)
(datetime.date(2026, 6, 23), datetime.date(2026, 6, 10), 9232405.0, 10127780.0, 19360185.0)
(datetime.date(2026, 7, 28), datetime.date(2026, 6, 10), 9825335.0, 8697390.0, 18522725.0)
(datetime.date(2026, 9, 29), datetime.date(2026, 6, 10), 5249010.0, 4500755.0, 9749765.0)
(datetime.date(2026, 8, 25), datetime.date(2026, 6, 10), 2054260.0, 1737060.0, 3791320.0)
(datetime.date(2026, 7, 7), datetime.date(2026, 6, 10), 275860.0, 505245.0, 781105.0)
(datetime.date(2027, 12, 28), datetime.date(2026, 6, 10), 218430.0, 155535.0, 373965.0)
(datetime.date(2028, 12, 26), datetime.date(2026, 6, 10), 44980.0, 16115.0, 61095.0)
(datetime.date(2026, 7, 14), datetime.date(2026, 6, 10), 9425.0, 12155.0, 21580.0)
(datetime.dat

In [11]:
results = con.execute("""
    SELECT
        percentile_cont(0.25) WITHIN GROUP (ORDER BY total_oi) AS p25,
        percentile_cont(0.50) WITHIN GROUP (ORDER BY total_oi) AS p50,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY total_oi) AS p75,
        percentile_cont(0.90) WITHIN GROUP (ORDER BY total_oi) AS p90,
        percentile_cont(0.95) WITHIN GROUP (ORDER BY total_oi) AS p95,
        MIN(total_oi)  AS min_oi,
        MAX(total_oi)  AS max_oi,
        COUNT(*)       AS total_rows
    FROM (
        SELECT pe_oi + ce_oi AS total_oi
        FROM nse.options_analytics
        WHERE ticker = 'NIFTY'
          AND pe_oi IS NOT NULL
          AND ce_oi IS NOT NULL
    )
""").fetchall()
for row in results: print(row)

(2875.0, 307227.5, 11629806.25, 84908952.50000001, 204765228.74999982, 0.0, 424741950.0, 8928)


In [12]:
results = con.execute("""
    SELECT
        trade_date,
        COUNT(*) AS liquid_expiries
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
    ORDER BY trade_date DESC
    LIMIT 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 22), 5)
(datetime.date(2026, 6, 19), 4)
(datetime.date(2026, 6, 18), 4)
(datetime.date(2026, 6, 17), 4)
(datetime.date(2026, 6, 16), 5)
(datetime.date(2026, 6, 15), 5)
(datetime.date(2026, 6, 12), 5)
(datetime.date(2026, 6, 11), 5)
(datetime.date(2026, 6, 10), 5)
(datetime.date(2026, 6, 9), 5)
(datetime.date(2026, 6, 8), 5)
(datetime.date(2026, 6, 5), 5)
(datetime.date(2026, 6, 4), 5)
(datetime.date(2026, 6, 3), 5)
(datetime.date(2026, 6, 2), 5)
(datetime.date(2026, 6, 1), 5)
(datetime.date(2026, 5, 29), 4)
(datetime.date(2026, 5, 27), 4)
(datetime.date(2026, 5, 26), 5)
(datetime.date(2026, 5, 25), 4)


In [13]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close              AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close             AS vix_close,
    pcr_agg.pcr,
    mp_agg.max_pain_dist_pct

FROM forward_returns fr

LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

-- PCR: all expiries
LEFT JOIN (
    SELECT
        trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: liquid expiries only, OI-weighted
LEFT JOIN (
    SELECT
        trade_date,
        ROUND(
            SUM(
                ((underlying - max_pain) / NULLIF(max_pain, 0) * 100)
                * (pe_oi + ce_oi)
            ) / NULLIF(SUM(pe_oi + ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [14]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 22), 24102.9, None, None, None, 0, 12.8425, 0.9654, -0.2197)
(datetime.date(2026, 6, 19), 24013.1, 0.374, None, None, 1, 12.97, 0.91, -0.1318)
(datetime.date(2026, 6, 18), 24168.0, -0.6409, None, None, 0, 12.6725, 1.1229, 0.2006)
(datetime.date(2026, 6, 17), 24085.7, 0.3417, None, None, 1, 13.1875, 1.1, 0.1268)
(datetime.date(2026, 6, 16), 23989.15, 0.4025, None, None, 1, 13.3625, 1.1117, 0.0237)
(datetime.date(2026, 6, 15), 23853.9, 0.567, 1.0439, None, 1, 14.3525, 0.9881, -0.1728)
(datetime.date(2026, 6, 12), 23622.9, 0.9779, 1.6518, None, 1, 14.7175, 1.412, -0.3175)
(datetime.date(2026, 6, 11), 23161.6, 1.9917, 4.3451, None, 1, 15.6125, 0.9853, -1.6451)
(datetime.date(2026, 6, 10), 23214.95, -0.2298, 3.7508, None, 0, 15.6325, 0.9344, -1.8159)
(datetime.date(2026, 6, 9), 23242.1, -0.1168, 3.2142, None, 0, 15.575, 0.9304, -0.9622)


In [15]:
results = con.execute("""
    SELECT
        CASE
            WHEN pcr < 0.7  THEN '1_very_low <0.7'
            WHEN pcr < 0.9  THEN '2_low 0.7-0.9'
            WHEN pcr < 1.1  THEN '3_neutral 0.9-1.1'
            WHEN pcr < 1.3  THEN '4_high 1.1-1.3'
            ELSE                 '5_very_high >1.3'
        END AS pcr_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND pcr IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("=== PCR ===")
for row in results: print(row)

results = con.execute("""
    SELECT
        CASE
            WHEN max_pain_dist_pct < -3   THEN '1_far_below <-3%'
            WHEN max_pain_dist_pct < -1.5 THEN '2_below -3 to -1.5%'
            WHEN max_pain_dist_pct < 0    THEN '3_slightly_below -1.5 to 0%'
            WHEN max_pain_dist_pct < 1.5  THEN '4_slightly_above 0 to 1.5%'
            ELSE                               '5_far_above >1.5%'
        END AS mp_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND max_pain_dist_pct IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("\n=== Max Pain Distance ===")
for row in results: print(row)

=== PCR ===
('1_very_low <0.7', 25, 0.025, 0.112, -1.021, 52.0)
('2_low 0.7-0.9', 177, -0.073, -0.005, 0.203, 42.4)
('3_neutral 0.9-1.1', 162, 0.038, 0.055, -0.032, 51.9)
('4_high 1.1-1.3', 108, 0.062, 0.088, -0.008, 57.4)
('5_very_high >1.3', 23, 0.157, -0.176, -1.076, 78.3)

=== Max Pain Distance ===
('1_far_below <-3%', 5, 0.292, 0.385, 5.101, 80.0)
('2_below -3 to -1.5%', 28, 0.099, 0.58, 1.701, 67.9)
('3_slightly_below -1.5 to 0%', 289, 0.001, 0.071, -0.203, 46.7)
('4_slightly_above 0 to 1.5%', 172, 0.007, -0.134, -0.151, 54.7)
('5_far_above >1.5%', 1, -1.39, 0.083, 0.716, 0.0)


In [16]:
# Futures - what tickers and how many rows
results = con.execute("""
    SELECT instrument_type, ticker, COUNT(*) as rows, 
           MIN(trade_date) as from_date, MAX(trade_date) as to_date
    FROM nse.futures_analytics
    GROUP BY instrument_type, ticker
    ORDER BY rows DESC
    LIMIT 10
""").fetchall()
print("=== Futures ===")
for row in results: print(row)

=== Futures ===
('STF', 'ULTRACEMCO', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('STF', 'BPCL', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('STF', 'ASIANPAINT', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('STF', 'HAL', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('STF', 'PNB', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('STF', 'HDFCAMC', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('STF', 'HDFCLIFE', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('IDF', 'NIFTY', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('STF', 'INDHOTEL', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))
('STF', 'DIVISLAB', 1488, datetime.date(2024, 6, 21), datetime.date(2026, 6, 22))


In [17]:
# Participant - what participant types and asset classes exist
results = con.execute("""
    SELECT participant_type, metric_type, asset_class, direction, option_side,
           COUNT(*) as rows
    FROM nse.participant_activity
    GROUP BY participant_type, metric_type, asset_class, direction, option_side
    ORDER BY participant_type, metric_type, asset_class
    LIMIT 30
""").fetchall()
print("\n=== Participant ===")
for row in results: print(row)


=== Participant ===
('Client', 'OI', 'INDEX', 'long', 'CE', 496)
('Client', 'OI', 'INDEX', 'long', 'PE', 496)
('Client', 'OI', 'INDEX', 'short', 'NA', 496)
('Client', 'OI', 'INDEX', 'short', 'PE', 496)
('Client', 'OI', 'INDEX', 'long', 'NA', 496)
('Client', 'OI', 'INDEX', 'short', 'CE', 496)
('Client', 'OI', 'STOCK', 'long', 'NA', 496)
('Client', 'OI', 'STOCK', 'long', 'PE', 496)
('Client', 'OI', 'STOCK', 'short', 'PE', 496)
('Client', 'OI', 'STOCK', 'short', 'CE', 496)
('Client', 'OI', 'STOCK', 'long', 'CE', 496)
('Client', 'OI', 'STOCK', 'short', 'NA', 496)
('Client', 'VOL', 'INDEX', 'long', 'CE', 496)
('Client', 'VOL', 'INDEX', 'short', 'PE', 496)
('Client', 'VOL', 'INDEX', 'short', 'NA', 496)
('Client', 'VOL', 'INDEX', 'long', 'NA', 496)
('Client', 'VOL', 'INDEX', 'long', 'PE', 496)
('Client', 'VOL', 'INDEX', 'short', 'CE', 496)
('Client', 'VOL', 'STOCK', 'long', 'NA', 496)
('Client', 'VOL', 'STOCK', 'short', 'NA', 496)
('Client', 'VOL', 'STOCK', 'long', 'CE', 496)
('Client', 'VOL

In [18]:
results = con.execute("""
    SELECT trade_date, expiry, basis, cost_of_carry, 
           chng_oi_per, open_int
    FROM nse.futures_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDF'
      AND trade_date = '2026-06-10'
    ORDER BY expiry ASC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 10), datetime.date(2026, 6, 30), 25.149999999997817, 0.019771203470175906, -1.4561936771066268, 19134310.0)
(datetime.date(2026, 6, 10), datetime.date(2026, 7, 28), 129.95000000000073, 0.042565737093267005, 0.20905923344947736, 1682460.0)
(datetime.date(2026, 6, 10), datetime.date(2026, 8, 25), 232.45000000000073, 0.04808848222918073, 1.3488880787458988, 542100.0)


In [19]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close             AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,

    -- VIX
    vix.close            AS vix_close,

    -- PCR (all expiries)
    pcr_agg.pcr,

    -- Max pain distance (liquid expiries, OI-weighted)
    mp_agg.max_pain_dist_pct,

    -- Futures near month
    fut.basis,
    fut.cost_of_carry,
    fut.chng_oi_per      AS fut_chng_oi_pct,

    -- FII net futures positioning (long - short, futures only)
    ROUND(
        (fii_long_fut.contracts - fii_short_fut.contracts) /
        NULLIF(fii_long_fut.contracts + fii_short_fut.contracts, 0) * 100
    , 2)                 AS fii_fut_net_pct,

    -- Client net futures positioning (retail, often contrarian)
    ROUND(
        (cli_long_fut.contracts - cli_short_fut.contracts) /
        NULLIF(cli_long_fut.contracts + cli_short_fut.contracts, 0) * 100
    , 2)                 AS client_fut_net_pct,

    -- FII put buying (protection signal)
    ROUND(
        (fii_long_pe.contracts - fii_short_pe.contracts) /
        NULLIF(fii_long_pe.contracts + fii_short_pe.contracts, 0) * 100
    , 2)                 AS fii_pe_net_pct

FROM forward_returns fr

-- VIX
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'India VIX'

-- PCR: all expiries
LEFT JOIN (
    SELECT trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY' AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: liquid expiries only, OI-weighted
LEFT JOIN (
    SELECT trade_date,
        ROUND(
            SUM(((underlying - max_pain) / NULLIF(max_pain, 0) * 100) * (pe_oi + ce_oi))
            / NULLIF(SUM(pe_oi + ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY' AND instrument_type = 'IDO'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

-- Futures: near month only
LEFT JOIN (
    SELECT trade_date, basis, cost_of_carry, chng_oi_per
    FROM nse.futures_analytics
    WHERE ticker = 'NIFTY' AND instrument_type = 'IDF'
      AND (trade_date, expiry) IN (
          SELECT trade_date, MIN(expiry)
          FROM nse.futures_analytics
          WHERE ticker = 'NIFTY' AND instrument_type = 'IDF'
          GROUP BY trade_date
      )
) fut ON fut.trade_date = fr.trade_date

-- Participant: FII long futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'NA'
) fii_long_fut ON fii_long_fut.trade_date = fr.trade_date

-- Participant: FII short futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'NA'
) fii_short_fut ON fii_short_fut.trade_date = fr.trade_date

-- Participant: Client long futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'Client' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'NA'
) cli_long_fut ON cli_long_fut.trade_date = fr.trade_date

-- Participant: Client short futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'Client' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'NA'
) cli_short_fut ON cli_short_fut.trade_date = fr.trade_date

-- Participant: FII long PE (put buying = hedging/bearish)
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'PE'
) fii_long_pe ON fii_long_pe.trade_date = fr.trade_date

-- Participant: FII short PE
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'PE'
) fii_short_pe ON fii_short_pe.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [20]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 5
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 22), 24102.9, None, None, None, 0, 12.8425, 0.9654, -0.2197, 20.899999999997817, 0.03956214812325074, -2.3829616076216915, -73.69, 50.95, 35.62)
(datetime.date(2026, 6, 19), 24013.1, 0.374, None, None, 1, 12.97, 0.91, -0.1318, 43.80000000000291, 0.060523782283992196, -0.16770465085151448, -74.09, 51.6, 37.34)
(datetime.date(2026, 6, 18), 24168.0, -0.6409, None, None, 0, 12.6725, 1.1229, 0.2006, 24.5, 0.030834505682445106, -1.7608316477872616, -73.05, 51.29, 35.21)
(datetime.date(2026, 6, 17), 24085.7, 0.3417, None, None, 1, 13.1875, 1.1, 0.1268, 8.299999999999272, 0.009675386704079228, -2.1750151331719128, -73.82, 50.99, 36.74)
(datetime.date(2026, 6, 16), 23989.15, 0.4025, None, None, 1, 13.3625, 1.1117, 0.0237, 11.849999999998545, 0.012878590053061098, -3.575780127096694, -74.58, 51.43, 39.58)


In [21]:
queries = {
    "Basis": ("basis", [
        ("1_discount <0",      "basis < 0"),
        ("2_low 0-50",         "basis >= 0 AND basis < 50"),
        ("3_mid 50-100",       "basis >= 50 AND basis < 100"),
        ("4_high >100",        "basis >= 100"),
    ]),
    "FII Fut Net": ("fii_fut_net_pct", [
        ("1_very_short <-50",  "fii_fut_net_pct < -50"),
        ("2_short -50 to -20", "fii_fut_net_pct >= -50 AND fii_fut_net_pct < -20"),
        ("3_neutral -20 to 20","fii_fut_net_pct >= -20 AND fii_fut_net_pct < 20"),
        ("4_long 20 to 50",    "fii_fut_net_pct >= 20 AND fii_fut_net_pct < 50"),
        ("5_very_long >50",    "fii_fut_net_pct >= 50"),
    ]),
    "Client Fut Net": ("client_fut_net_pct", [
        ("1_very_short <-50",  "client_fut_net_pct < -50"),
        ("2_short -50 to -20", "client_fut_net_pct >= -50 AND client_fut_net_pct < -20"),
        ("3_neutral -20 to 20","client_fut_net_pct >= -20 AND client_fut_net_pct < 20"),
        ("4_long 20 to 50",    "client_fut_net_pct >= 20 AND client_fut_net_pct < 50"),
        ("5_very_long >50",    "client_fut_net_pct >= 50"),
    ]),
    "FII PE Net": ("fii_pe_net_pct", [
        ("1_very_short <-50",  "fii_pe_net_pct < -50"),
        ("2_short -50 to -20", "fii_pe_net_pct >= -50 AND fii_pe_net_pct < -20"),
        ("3_neutral -20 to 20","fii_pe_net_pct >= -20 AND fii_pe_net_pct < 20"),
        ("4_long 20 to 50",    "fii_pe_net_pct >= 20 AND fii_pe_net_pct < 50"),
        ("5_very_long >50",    "fii_pe_net_pct >= 50"),
    ]),
}

for factor_name, (col, buckets) in queries.items():
    case_sql = "CASE\n" + "\n".join(
        f"  WHEN {cond} THEN '{label}'" for label, cond in buckets
    ) + "\n  ELSE 'other' END"

    results = con.execute(f"""
        SELECT
            {case_sql} AS bucket,
            COUNT(*)                    AS days,
            ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
            ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
            ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
            ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days
        FROM daily_features
        WHERE fwd_ret_1d IS NOT NULL AND {col} IS NOT NULL
        GROUP BY 1 ORDER BY 1
    """).fetchall()

    print(f"\n=== {factor_name} ===")
    for row in results: print(row)


=== Basis ===
('1_discount <0', 47, 0.093, 0.674, 0.218, 59.6)
('2_low 0-50', 160, 0.106, 0.174, 0.21, 54.4)
('3_mid 50-100', 194, -0.07, -0.098, -0.15, 47.4)
('4_high >100', 94, -0.037, -0.252, -0.361, 47.9)

=== FII Fut Net ===
('1_very_short <-50', 322, -0.01, 0.031, -0.036, 49.4)
('2_short -50 to -20', 76, 0.044, -0.261, -0.414, 52.6)
('3_neutral -20 to 20', 43, 0.108, 0.618, 0.945, 53.5)
('4_long 20 to 50', 30, -0.003, 0.253, 0.493, 60.0)
('5_very_long >50', 24, -0.022, -0.343, -1.32, 50.0)

=== Client Fut Net ===
('2_short -50 to -20', 29, 0.047, -0.091, -1.386, 55.2)
('3_neutral -20 to 20', 108, 0.073, 0.385, 1.365, 55.6)
('4_long 20 to 50', 331, -0.025, -0.175, -0.48, 48.0)
('5_very_long >50', 27, 0.122, 1.528, 2.355, 63.0)

=== FII PE Net ===
('3_neutral -20 to 20', 212, -0.015, -0.133, -0.618, 50.9)
('4_long 20 to 50', 282, 0.028, 0.156, 0.422, 51.1)
('5_very_long >50', 1, -0.302, 0.953, 1.592, 0.0)


In [22]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL

    GROUP BY 1, 2
    ORDER BY 1, 2
""").fetchall()

for row in results: print(row)

('VIX_high', 'basis_discount', 11, -0.486, 1.073, 1.22, 54.5)
('VIX_high', 'basis_high', 6, 0.082, 1.017, 0.964, 66.7)
('VIX_high', 'basis_low', 25, 0.375, 0.278, 1.399, 56.0)
('VIX_high', 'basis_mid', 22, 0.226, 1.195, 2.519, 59.1)
('VIX_low', 'basis_discount', 15, 0.28, 0.738, -0.683, 66.7)
('VIX_low', 'basis_high', 62, -0.085, -0.41, -0.297, 45.2)
('VIX_low', 'basis_low', 79, 0.004, -0.059, -0.445, 54.4)
('VIX_low', 'basis_mid', 106, -0.045, -0.114, -0.337, 52.8)
('VIX_mid', 'basis_discount', 21, 0.262, 0.419, 0.342, 57.1)
('VIX_mid', 'basis_high', 26, 0.048, -0.169, -0.879, 50.0)
('VIX_mid', 'basis_low', 56, 0.131, 0.439, 0.577, 53.6)
('VIX_mid', 'basis_mid', 66, -0.21, -0.504, -0.82, 34.8)


In [23]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        CASE
            WHEN fii_fut_net_pct < -50 THEN 'FII_short'
            WHEN fii_fut_net_pct < 20  THEN 'FII_neutral'
            ELSE                            'FII_long'
        END AS fii_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND fii_fut_net_pct IS NOT NULL

    GROUP BY 1, 2, 3
    HAVING COUNT(*) >= 8      -- filter cells too small to mean anything
    ORDER BY avg_ret_20d DESC
""").fetchall()

for row in results: print(row)

('VIX_high', 'basis_mid', 'FII_neutral', 8, 0.996, 1.524, 3.099, 87.5)
('VIX_high', 'basis_mid', 'FII_short', 14, -0.214, 1.007, 2.188, 42.9)
('VIX_low', 'basis_low', 'FII_neutral', 15, 0.106, -0.135, 1.662, 66.7)
('VIX_high', 'basis_low', 'FII_short', 20, 0.544, 0.376, 1.314, 65.0)
('VIX_high', 'basis_discount', 'FII_short', 10, -0.425, 0.855, 1.145, 60.0)
('VIX_mid', 'basis_low', 'FII_neutral', 27, 0.185, 0.818, 0.852, 63.0)
('VIX_low', 'basis_mid', 'FII_long', 14, 0.283, -0.027, 0.745, 78.6)
('VIX_mid', 'basis_low', 'FII_short', 22, 0.081, 0.046, 0.516, 40.9)
('VIX_low', 'basis_mid', 'FII_short', 72, -0.046, 0.17, 0.067, 51.4)
('VIX_low', 'basis_high', 'FII_short', 49, 0.008, 0.036, -0.113, 49.0)
('VIX_mid', 'basis_high', 'FII_short', 17, 0.078, -0.361, -0.205, 47.1)
('VIX_mid', 'basis_discount', 'FII_short', 9, 0.553, -0.09, -0.532, 77.8)
('VIX_mid', 'basis_mid', 'FII_short', 46, -0.229, -0.61, -0.645, 34.8)
('VIX_low', 'basis_low', 'FII_short', 54, -0.044, -0.066, -0.884, 50.0)
('

In [24]:
import scipy.stats as stats
import pandas as pd

df = con.execute("""
    SELECT vix_close, pcr, max_pain_dist_pct, basis, 
           cost_of_carry, fut_chng_oi_pct,
           fii_fut_net_pct, client_fut_net_pct,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
""").df()

factors = ['vix_close', 'pcr', 'max_pain_dist_pct', 'basis',
           'cost_of_carry', 'fut_chng_oi_pct', 'fii_fut_net_pct', 
           'client_fut_net_pct']

targets = ['fwd_ret_1d', 'fwd_ret_5d', 'fwd_ret_20d']

rows = []
for f in factors:
    row = {'factor': f}
    for t in targets:
        mask = df[f].notna() & df[t].notna()
        ic, pval = stats.spearmanr(df.loc[mask, f], df.loc[mask, t])
        row[f'IC_{t}'] = round(ic, 4)
        row[f'pval_{t}'] = round(pval, 4)
    rows.append(row)

ic_df = pd.DataFrame(rows)
print(ic_df.to_string(index=False))

            factor  IC_fwd_ret_1d  pval_fwd_ret_1d  IC_fwd_ret_5d  pval_fwd_ret_5d  IC_fwd_ret_20d  pval_fwd_ret_20d
         vix_close         0.0606           0.1783         0.0856           0.0579          0.1553            0.0007
               pcr         0.0755           0.0933         0.0485           0.2830         -0.0189            0.6806
 max_pain_dist_pct        -0.0014           0.9755        -0.0063           0.8888         -0.0672            0.1431
             basis        -0.0975           0.0300        -0.1344           0.0028         -0.0936            0.0413
     cost_of_carry        -0.0354           0.4440        -0.0889           0.0550          0.0062            0.8956
   fut_chng_oi_pct        -0.0742           0.0993        -0.0983           0.0295         -0.0547            0.2340
   fii_fut_net_pct        -0.0157           0.7269         0.0083           0.8553         -0.0372            0.4178
client_fut_net_pct         0.0189           0.6746         0.020

In [25]:
# IC of max pain dist conditioned on being away from max pain
mask_tail = df['max_pain_dist_pct'].abs() > 1.5
sub = df[mask_tail & df['fwd_ret_20d'].notna() & df['max_pain_dist_pct'].notna()]
ic, pval = stats.spearmanr(sub['max_pain_dist_pct'], sub['fwd_ret_20d'])
print(f"Max pain tail IC (20D): {ic:.4f}, p={pval:.4f}, n={len(sub)}")

Max pain tail IC (20D): -0.5753, p=0.0014, n=28


In [26]:
TRAIN_END = '2025-06-20'
TEST_START = '2025-06-21'

# Verify the split
results = con.execute(f"""
    SELECT 
        CASE WHEN trade_date <= '{TRAIN_END}' THEN 'train' ELSE 'test' END AS split,
        COUNT(*) as days,
        MIN(trade_date) as from_date,
        MAX(trade_date) as to_date
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()
for row in results: print(row)

('test', 246, datetime.date(2025, 6, 23), datetime.date(2026, 6, 19))
('train', 249, datetime.date(2024, 6, 21), datetime.date(2025, 6, 20))


In [27]:
con.execute(f"""
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS split VARCHAR;
    
    UPDATE daily_features 
    SET split = CASE 
        WHEN trade_date <= '{TRAIN_END}' THEN 'train' 
        ELSE 'test' 
    END
""")

In [28]:
df_train = con.execute("""
    SELECT vix_close, pcr, max_pain_dist_pct, basis, 
           cost_of_carry, fut_chng_oi_pct,
           fii_fut_net_pct, client_fut_net_pct,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND split = 'train'
""").df()

rows = []
for f in factors:
    row = {'factor': f}
    for t in targets:
        mask = df_train[f].notna() & df_train[t].notna()
        ic, pval = stats.spearmanr(df_train.loc[mask, f], df_train.loc[mask, t])
        row[f'IC_{t}'] = round(ic, 4)
        row[f'pval_{t}'] = round(pval, 4)
        row[f'n_{t}'] = mask.sum()
    rows.append(row)

ic_df_train = pd.DataFrame(rows)
print("=== IC Analysis — TRAIN ONLY (Jun 2024 - Jun 2025) ===")
print(ic_df_train.to_string(index=False))

=== IC Analysis — TRAIN ONLY (Jun 2024 - Jun 2025) ===
            factor  IC_fwd_ret_1d  pval_fwd_ret_1d  n_fwd_ret_1d  IC_fwd_ret_5d  pval_fwd_ret_5d  n_fwd_ret_5d  IC_fwd_ret_20d  pval_fwd_ret_20d  n_fwd_ret_20d
         vix_close         0.0855           0.1787           249         0.2049           0.0011           249          0.1934            0.0022            249
               pcr         0.1255           0.0480           249         0.0869           0.1715           249          0.0168            0.7922            249
 max_pain_dist_pct         0.0445           0.4843           249        -0.0243           0.7022           249          0.0261            0.6814            249
             basis        -0.0914           0.1504           249        -0.2459           0.0001           249         -0.1198            0.0591            249
     cost_of_carry        -0.0612           0.3483           237        -0.1717           0.0081           237         -0.1143            0.0790 

In [29]:
mask_tail = df_train['max_pain_dist_pct'].abs() > 1.5
sub = df_train[mask_tail & df_train['fwd_ret_20d'].notna() & df_train['max_pain_dist_pct'].notna()]
ic, pval = stats.spearmanr(sub['max_pain_dist_pct'], sub['fwd_ret_20d'])
print(f"\nMax pain tail IC 20D (train only): {ic:.4f}, p={pval:.4f}, n={len(sub)}")


Max pain tail IC 20D (train only): -0.1786, p=0.7017, n=7


In [30]:
results = con.execute("""
    SELECT trade_date, max_pain_dist_pct, fwd_ret_20d, split, vix_close
    FROM daily_features
    WHERE ABS(max_pain_dist_pct) > 1.5
      AND fwd_ret_20d IS NOT NULL
    ORDER BY trade_date
""").fetchall()
for row in results: print(row)

(datetime.date(2024, 10, 7), -1.5476, -3.228, 'train', 15.08)
(datetime.date(2024, 10, 25), -1.6114, 0.0567, 'train', 14.6325)
(datetime.date(2024, 12, 20), -1.8952, -1.0291, 'train', 15.0725)
(datetime.date(2025, 1, 13), -1.5501, 2.0532, 'train', 15.9975)
(datetime.date(2025, 1, 27), -1.6451, -0.1456, 'train', 18.1325)
(datetime.date(2025, 4, 7), -2.4887, 8.3315, 'train', 22.7925)
(datetime.date(2025, 5, 12), 1.6132, 0.7162, 'train', 18.3925)
(datetime.date(2026, 3, 4), -2.5576, -6.1774, 'test', 21.14)
(datetime.date(2026, 3, 6), -1.7535, -1.8531, 'test', 19.88)
(datetime.date(2026, 3, 9), -1.6644, -1.0527, 'test', 23.3625)
(datetime.date(2026, 3, 11), -2.7409, -0.1014, 'test', 21.0625)
(datetime.date(2026, 3, 12), -3.0533, 2.505, 'test', 21.5175)
(datetime.date(2026, 3, 13), -3.3119, 4.5166, 'test', 22.645)
(datetime.date(2026, 3, 16), -1.8112, 4.0359, 'test', 21.6025)
(datetime.date(2026, 3, 18), -1.865, 3.3594, 'test', 18.7225)
(datetime.date(2026, 3, 19), -3.6389, 5.9818, 'test', 

In [31]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND split = 'train'
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL

    GROUP BY 1, 2
    HAVING COUNT(*) >= 5
    ORDER BY 1, 2
""").fetchall()

print("=== VIX × Basis (TRAIN ONLY) ===")
for row in results: print(row)

=== VIX × Basis (TRAIN ONLY) ===
('VIX_high', 'basis_low', 7, -0.007, 0.579, 0.968, 42.9)
('VIX_high', 'basis_mid', 10, 0.928, 2.574, 4.512, 80.0)
('VIX_low', 'basis_discount', 11, 0.26, 0.939, -0.153, 63.6)
('VIX_low', 'basis_high', 16, -0.243, -1.225, 0.444, 31.3)
('VIX_low', 'basis_low', 27, 0.044, 0.133, 0.897, 63.0)
('VIX_low', 'basis_mid', 38, -0.1, -0.407, -0.458, 52.6)
('VIX_mid', 'basis_discount', 16, 0.343, 1.171, 1.401, 56.3)
('VIX_mid', 'basis_high', 21, 0.181, 0.211, -0.371, 57.1)
('VIX_mid', 'basis_low', 46, 0.067, 0.46, 0.948, 50.0)
('VIX_mid', 'basis_mid', 53, -0.199, -0.522, -0.517, 34.0)


In [32]:
con.close()